# Week 4 — Supervised Learning Model Implementation
**Dataset:** Breast Cancer Wisconsin (Diagnostic), via scikit-learn

Binary classification (malignant vs benign) using Logistic Regression, validated with 5-fold stratified cross-validation. Full write-up in `Week_4_Report.docx`. Actual results: 98.25% test accuracy, ROC-AUC 0.995.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

## 2. Load and Inspect Data

In [2]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

print("Shape:", df.shape)
print("Missing values total:", df.isnull().sum().sum())
print("Target counts:\n", df["target"].value_counts())
print("Target names:", data.target_names)

Shape: (569, 31)
Missing values total: 0
Target counts:
 target
1    357
0    212
Name: count, dtype: int64
Target names: ['malignant' 'benign']


## 3. Train/Test Split and Pipeline
Scaling is placed inside a `Pipeline` so it's fit only on training folds — no leakage during cross-validation.

In [3]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=42))
])

## 4. Train and Cross-Validate

In [4]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_accuracy = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")

print("CV Accuracy per fold:", np.round(cv_accuracy, 4))
print("Mean CV Accuracy:", round(cv_accuracy.mean(), 4), "std:", round(cv_accuracy.std(), 4))
print("Mean CV F1:", round(cv_f1.mean(), 4))

CV Accuracy per fold: [0.967 0.989 0.978 0.989 0.967]
Mean CV Accuracy: 0.978 std: 0.0098
Mean CV F1: 0.9825


## 5. Test Set Evaluation

In [5]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Test Accuracy:", round(accuracy, 4))
print("Test Precision:", round(precision, 4))
print("Test Recall:", round(recall, 4))
print("Test F1:", round(f1, 4))
print("Test ROC-AUC:", round(roc_auc, 4))
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

Test Accuracy: 0.9825
Test Precision: 0.9861
Test Recall: 0.9861
Test F1: 0.9861
Test ROC-AUC: 0.9954

              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## 6. Confusion Matrix

In [6]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[41  1]
 [ 1 71]]


## 7. Model Interpretation — Coefficients

In [7]:
classifier = model.named_steps["classifier"]
coefficients = pd.Series(classifier.coef_[0], index=X.columns).sort_values()

print("Top 5 coefficients toward malignant:\n", coefficients.head(5))
print("\nTop 5 coefficients toward benign:\n", coefficients.tail(5))

Top 5 coefficients toward malignant:
 worst texture          -1.255088
radius error           -1.082965
worst concave points   -0.953686
worst area             -0.947756
worst radius           -0.947616
dtype: float64

Top 5 coefficients toward benign:
 texture error              0.248823
symmetry error             0.360492
fractal dimension error    0.437894
compactness error          0.647227
mean compactness           0.648342
dtype: float64


## Conclusion
98.25% test accuracy, ROC-AUC 0.995, only 2 misclassifications out of 114 test cases (1 false negative, 1 false positive). See `Week_4_Report.docx` for the full discussion of why the false negative matters more than the aggregate accuracy suggests.